## 這頁在做什麼

一個方波看起來只有高低兩段，但它其實可以由一堆不同頻率的正弦波疊出來。這頁用「轉動的圓」把這件事畫出來。每個圓代表一個頻率，半徑是振幅，圓心接在下一個圓上。把最外圈的點連起來，就會得到逼近的方波。

先看結果。下面四列分別用 1、2、3、4 個奇次諧波逼近方波。列數越多，波形越接近方塊，但邊緣的振鈴不會完全消失。

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

for base in (Path.cwd(), Path.cwd().parent, Path.cwd() / "scripts"):
    if (base / "notebook_media.py").exists():
        sys.path.insert(0, str(base))
        break

from notebook_media import ensure_ffmpeg
ensure_ffmpeg()

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from matplotlib.animation import FuncAnimation

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 120,
    "font.sans-serif": ["PingFang TC", "Heiti TC", "Arial Unicode MS", "DejaVu Sans"],
    "axes.unicode_minus": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

TAU = 2 * np.pi

In [ ]:
def epicycle_state(angle, n_terms, kind="square"):
    # Return harmonic terms, radii, and the tip-to-tail points.
    if kind == "square":
        terms = [2 * k + 1 for k in range(n_terms)]
    elif kind == "sawtooth":
        terms = list(range(1, n_terms + 1))
    else:
        raise ValueError(kind)

    x = y = 0.0
    points = [(0.0, 0.0)]
    radii = []

    for n in terms:
        radius = (4 if kind == "square" else 2) / (n * np.pi)
        phase = 0.0 if kind == "square" else (np.pi if n % 2 == 1 else 0.0)
        x += radius * np.cos(n * angle + phase)
        y += radius * np.sin(n * angle + phase)
        radii.append(radius)
        points.append((x, y))

    return terms, radii, points

## 先看結果：四列逐項逼近

每一列多一個奇次諧波。左邊是圓的接力，右邊是接力終點留下的波形。

In [ ]:
def animate_wave(kind, filename, poster):
    rows = 4
    frames = 240
    angles = np.linspace(0, TAU, frames)
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

    fig, axes = plt.subplots(rows, 1, figsize=(10, 8), constrained_layout=True)
    circles, arms, waves = [], [], []

    for i, ax in enumerate(axes):
        terms, radii, _ = epicycle_state(0.0, i + 1, kind)
        row_circles = []
        for j, radius in enumerate(radii):
            circle = Circle((0, 0), radius, fill=False, color=colors[j], alpha=0.5, lw=0.8)
            ax.add_patch(circle)
            row_circles.append(circle)
        arm, = ax.plot([], [], "o-", color=colors[i], lw=1, ms=2)
        wave, = ax.plot([], [], color=colors[i], lw=1.5)
        ax.set_xlim(-1.35, 5.2)
        ax.set_ylim(-1.35, 1.35)
        ax.set_aspect("equal")
        ax.axis("off")
        ax.text(-1.3, 0, f"n={', '.join(str(n) for n in terms)}",
                fontsize=10, va="center", ha="left")
        circles.append(row_circles)
        arms.append(arm)
        waves.append(wave)

    history = [[] for _ in range(rows)]

    def update(frame):
        angle = angles[frame]
        artists = []
        for i in range(rows):
            terms, radii, points = epicycle_state(angle, i + 1, kind)
            for j, (radius, center) in enumerate(zip(radii, points[:-1])):
                circles[i][j].set_center(center)
                circles[i][j].set_radius(radius)
            arms[i].set_data([p[0] for p in points], [p[1] for p in points])
            history[i].insert(0, points[-1][1])
            if len(history[i]) > frames:
                history[i].pop()
            xs = np.linspace(1.55, 1.55 + len(history[i]) * 0.012, len(history[i]))
            waves[i].set_data(xs, history[i])
            artists.extend([arms[i], waves[i]] + circles[i])
        return artists

    anim = FuncAnimation(fig, update, frames=frames, interval=50, blit=True)
    for i in range(frames):
        update(i)
    fig.savefig(poster, dpi=120, bbox_inches="tight")
    anim.save(
        filename,
        writer="ffmpeg",
        fps=20,
        codec="libx264",
        extra_args=["-pix_fmt", "yuv420p", "-crf", "23"],
    )
    plt.close(fig)

In [ ]:
animate_wave(
    "square",
    "square_wave_increasing_terms.mp4",
    "square_wave_increasing_terms.png",
)

## 一個圓的投影就是正弦波

把一個點放在半徑 1 的圓上，讓它等速旋轉。這個點的 $y$ 座標隨時間變化，就是一條正弦波。圓的半徑是振幅，轉一圈的時間決定頻率，起始角度決定相位。

In [ ]:
theta = np.linspace(0, TAU, 400)
angle = np.pi / 4

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)

axes[0].plot(np.cos(theta), np.sin(theta), color="tab:blue")
axes[0].plot([0, np.cos(angle)], [0, np.sin(angle)], color="tab:red")
axes[0].plot([np.cos(angle)], [np.sin(angle)], "o", color="tab:red")
axes[0].set_aspect("equal")
axes[0].set_xlim(-1.2, 1.2)
axes[0].set_ylim(-1.2, 1.2)
axes[0].set_title("等速圓周運動")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

axes[1].plot(theta, np.sin(theta), color="tab:blue")
axes[1].plot([angle], [np.sin(angle)], "o", color="tab:red")
axes[1].axhline(0, color="0.5", lw=0.8)
axes[1].set_title("投影到 y 軸就是正弦波")
axes[1].set_xlabel("角度 [rad]")
axes[1].set_ylabel("sin(theta)")

## 把多個圓接起來

把第二個圓的圓心接到第一個圓的終點，再把第三個圓接到第二個圓的終點。最外圈的點會走出比較複雜的軌跡。這些圓的轉速分別是基頻的 1、3、5、7 倍，半徑則是 $4/(n\pi)$。

下圖把四個不同項數的接力畫在同一個角度。圓越多，終點越接近方波的邊緣。

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(9, 8), constrained_layout=True)
colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]
angle = 0.9

for i, ax in enumerate(axes):
    terms, radii, points = epicycle_state(angle, i + 1, "square")
    for j, (radius, center) in enumerate(zip(radii, points[:-1])):
        ax.add_patch(Circle(center, radius, fill=False, color=colors[j], alpha=0.5, lw=0.8))
    ax.plot([p[0] for p in points], [p[1] for p in points], "o-", color=colors[i], lw=1, ms=2)
    ax.plot([points[-1][0], 1.55], [points[-1][1]] * 2, "--", color="0.4", lw=0.8)
    ax.set_xlim(-1.3, 1.7)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"n={', '.join(str(n) for n in terms)}", loc="left", fontsize=10)

## 方波為什麼只有奇次諧波

方波是奇函數，而且左右對稱。把這個對稱性放進傅立葉級數，偶次諧波會互相抵銷，只剩下 1、3、5、7 這些奇次項。

方波的級數可以寫成：

$$
f(t)=\frac{4}{\pi}\left(\sin t+\frac{\sin 3t}{3}+\frac{\sin 5t}{5}+\cdots\right)
$$

每一項的振幅是 $4/(n\pi)$，所以頻率越高，振幅越小。這也是為什麼只取前幾項時，波形已經有方塊的輪廓。

## 從 1 項到 7 項

下圖把部分和畫出來。虛線是理想的方波，彩色線是不同項數的近似。項數增加時，平頂越來越平，邊緣越來越陡。

In [ ]:
def square_partial(t, n_terms):
    y = np.zeros_like(t)
    for k in range(n_terms):
        n = 2 * k + 1
        y += 4 / (n * np.pi) * np.sin(n * t)
    return y

t = np.linspace(0, TAU, 2000)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True, constrained_layout=True)
for i, ax in enumerate(axes):
    terms = [2 * k + 1 for k in range(i + 1)]
    y = square_partial(t, i + 1)
    ax.plot(t, y, color="tab:blue", lw=1.4, label=f"{len(terms)} terms")
    ax.axhline(1, color="0.5", ls="--", lw=0.8)
    ax.axhline(-1, color="0.5", ls="--", lw=0.8)
    ax.set_ylim(-1.5, 1.5)
    ax.set_ylabel("amplitude")
    ax.set_title(f"n={', '.join(str(n) for n in terms)}", loc="left", fontsize=10)
    ax.legend(loc="upper right", fontsize=8)
axes[-1].set_xlabel("angle [rad]")

## 邊緣的振鈴：Gibbs phenomenon

項數增加不會讓邊緣的過衝消失。過衝會變窄，但高度仍然存在，約為理想值的 9%。這個現象叫 Gibbs phenomenon。

下圖把邊緣放大。項數從 5 增加到 50，過衝的位置越來越靠近跳變點，但峰值幾乎不變。

In [ ]:
t = np.linspace(0, TAU, 5000)

fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
for n_terms, color in [(5, "tab:blue"), (15, "tab:orange"), (50, "tab:green")]:
    ax.plot(t, square_partial(t, n_terms), color=color, lw=1.2, label=f"{n_terms} terms")

ax.axhline(1.0895, color="0.4", ls=":", lw=1)
ax.text(0.08, 1.115, "Gibbs overshoot ≈ 1.0895", fontsize=9, color="0.3")
ax.set_xlim(0, np.pi / 2)
ax.set_ylim(0.6, 1.25)
ax.set_xlabel("angle [rad]")
ax.set_ylabel("amplitude")
ax.legend(loc="lower right", fontsize=9)

## 換成鋸齒波

鋸齒波用所有整數諧波，不是只有奇次。係數換成 $2/(n\pi)$，而且每隔一項要加 $180^\circ$ 的相位。相位交錯讓波形往同一個方向傾斜。

下面這支動畫把方波換成鋸齒波。程式只改了三個地方：諧波清單、半徑公式、相位。

In [ ]:
animate_wave(
    "sawtooth",
    "sawtooth_wave_approximation.mp4",
    "sawtooth_wave_approximation.png",
)

## 程式實作的三個關鍵

`epicycle_state()` 只做一件事：給定角度與項數，回傳每個圓的半徑、圓心與終點。三個關鍵參數是：

- 半徑：方波用 $4/(n\pi)$，鋸齒波用 $2/(n\pi)$。
- 頻率：方波只取奇數 $n$，鋸齒波取所有整數。
- 相位：鋸齒波在奇數項加 $\pi$，讓正負號交錯。

`animate_wave()` 把每個角度算出的終點放進 `history`，再把 `history` 畫成右側波形。動畫每一幀只更新圓心、半徑、手臂與波形，不需要重新計算整條曲線。

## 可以自己改的參數

- `n_terms`：每一列要用幾個諧波。項數越多，逼近越好，但計算越慢。
- `kind`：`"square"` 或 `"sawtooth"`。
- `frames`：動畫幀數。240 幀、20 fps 大約 12 秒。
- `figsize` 與 `dpi`：輸出尺寸。這頁用 1200×960。
- `crf`：影片品質。數字越小，檔案越大。

改完 `animate_wave()` 的參數後，重新執行呼叫的 cell，就會覆寫 mp4 與 poster。

## 小結

- 圓的投影是正弦波。半徑是振幅，轉速是頻率，起始角是相位。
- 把多個圓接起來，就是多個頻率的疊加。
- 方波只有奇次諧波，鋸齒波用所有整數諧波。
- 項數增加會讓波形更接近目標，但 Gibbs 過衝不會消失。

同樣的「把訊號拆成頻率」的想法，可以用來看機器震動。下一篇〈[一段震動訊號的五種看法](/notebooks/vibration-signal-five-views/)〉從時間域找事件。〈[馬達的八種壞法](/notebooks/motor-fault-spectra/)〉則把訊號轉成頻譜，看不同故障留下的指紋。